# AnemiaFusionNet — Phase 6: Evaluation

**Project:** AI-Powered Multimodal Anemia Detection System
**Phase:** 6 of 6 — Evaluation, Ablation & Reporting

---

### What this phase delivers

| Section | Content |
|---------|---------|
| 1 | Setup — imports, paths, device (same auto-detection as Phases 2-5) |
| 2 | Re-load Phase 5 architecture classes and best checkpoint |
| 3 | Full model evaluation on the held-out test set |
| 4 | Ablation study — three single-modality baselines (image-only, clinical-only, no-geo) |
| 5 | Modality contribution comparison table and bar chart |
| 6 | Confusion matrix |
| 7 | ROC curve with all baselines overlaid |
| 8 | Precision-Recall curve with all baselines |
| 9 | Classification report (CSV) |
| 10 | Calibration analysis (reliability diagram) |
| 11 | Error analysis — hardest false positives / false negatives |
| 12 | Final summary report |

**Runs independently** — Phase 5 does not need to be open. All checkpoints
and data are read from disk.


---
## Section 1 — Setup & Imports

In [1]:
# =============================================================================
#  1.1  Auto-install tqdm if needed (mirrors Phase 5 convention)
# =============================================================================
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("tqdm")

# =============================================================================
#  1.2  Standard library
# =============================================================================
import os
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# =============================================================================
#  1.3  Plotting
# =============================================================================
import matplotlib
matplotlib.use("Agg")   # non-interactive — safe on every machine / Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# =============================================================================
#  1.4  PyTorch
# =============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.models as tv_models
from torchvision import transforms

# =============================================================================
#  1.5  Scikit-learn
# =============================================================================
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report,
    matthews_corrcoef, brier_score_loss,
)
from sklearn.calibration import calibration_curve

from tqdm.auto import tqdm
from pathlib import Path

print(f"PyTorch   : {torch.__version__}")
print(f"NumPy     : {np.__version__}")
print(f"Pandas    : {pd.__version__}")


PyTorch   : 2.12.1+cpu
NumPy     : 2.2.4
Pandas    : 2.2.3


In [2]:
# =============================================================================
#  1.6  Project paths — same auto-detection as Phases 2-5
# =============================================================================
_cwd = Path.cwd()
if   (_cwd / "dataset").exists():        PROJECT_ROOT = _cwd
elif (_cwd.parent / "dataset").exists(): PROJECT_ROOT = _cwd.parent
else:                                     PROJECT_ROOT = _cwd   # graceful fallback

DATASET_DIR   = PROJECT_ROOT / "dataset"
PROCESSED_DIR = DATASET_DIR  / "processed"
IMAGE_DIR     = DATASET_DIR  / "images"
MODELS_DIR    = PROJECT_ROOT / "models"
OUTPUTS_DIR   = PROJECT_ROOT / "outputs"
REPORTS_DIR   = PROJECT_ROOT / "reports"

for _d in [OUTPUTS_DIR, REPORTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"Models dir   : {MODELS_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")


Project root : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet
Device       : cpu
Models dir   : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\models
Outputs dir  : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs


In [3]:
# =============================================================================
#  1.7  Global evaluation config — single source of truth
# =============================================================================
EVAL_CONFIG = {
    "batch_size"         : 16,
    "seed"               : 42,
    "threshold"          : 0.5,      # decision threshold for binary prediction
    "calibration_bins"   : 10,       # bins for reliability diagram
    "top_errors_n"       : 20,       # number of hardest errors to inspect

    # Checkpoint file names (must match what Phase 5 saved)
    "best_ckpt"          : "best_multimodal_model.pth",

    # Clinical feature count  (must match Phase 3/5 training schema)
    "required_clinical_dim" : 4,
}

SEED = EVAL_CONFIG["seed"]
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Evaluation configuration:")
for k, v in EVAL_CONFIG.items():
    print(f"  {k:<24} : {v}")


Evaluation configuration:
  batch_size               : 16
  seed                     : 42
  threshold                : 0.5
  calibration_bins         : 10
  top_errors_n             : 20
  best_ckpt                : best_multimodal_model.pth
  required_clinical_dim    : 4


---
## Section 2 — Reload Architecture, Data & Best Checkpoint

All class definitions are copied verbatim from Phase 5 so that `state_dict`
keys match exactly. The best checkpoint (`best_multimodal_model.pth`) saved
by Phase 5 Stage 3 is loaded here.


In [4]:
# =============================================================================
#  2.1  Architecture class definitions — MUST match Phase 5 / Phase 3 exactly
# =============================================================================

# ─── Image Feature Extractor ──────────────────────────────────────────────────
class ImageFeatureExtractor(nn.Module):
    """EfficientNet-B0 (or ResNet-50) backbone with classifier head removed."""

    SUPPORTED = {
        "EfficientNet-B0": {
            "builder"     : lambda w: tv_models.efficientnet_b0(weights=w),
            "weights_cls" : tv_models.EfficientNet_B0_Weights.DEFAULT,
            "feature_dim" : 1280,
        },
        "ResNet-50": {
            "builder"     : lambda w: tv_models.resnet50(weights=w),
            "weights_cls" : tv_models.ResNet50_Weights.DEFAULT,
            "feature_dim" : 2048,
        },
    }

    def __init__(self, backbone_name: str = "EfficientNet-B0", pretrained: bool = True):
        super().__init__()
        spec    = self.SUPPORTED[backbone_name]
        weights = spec["weights_cls"] if pretrained else None
        base    = spec["builder"](weights)
        self.feature_dim   = spec["feature_dim"]
        self.backbone_name = backbone_name

        if backbone_name == "EfficientNet-B0":
            for p in base.parameters():
                p.requires_grad = False
            for block in list(base.features.children())[-2:]:
                for p in block.parameters():
                    p.requires_grad = True
            self.backbone = base.features
            self.pool     = base.avgpool
        else:  # ResNet-50
            for p in base.parameters():
                p.requires_grad = False
            for layer in [base.layer3, base.layer4]:
                for p in layer.parameters():
                    p.requires_grad = True
            self.backbone = nn.Sequential(*list(base.children())[:-1])
            self.pool     = nn.Identity()

    def forward(self, x):
        return self.pool(self.backbone(x)).flatten(1)


# ─── Clinical Feature Extractor ───────────────────────────────────────────────
class ClinicalFeatureExtractor(nn.Module):
    """MLP: input_dim → 256 → 128 → 64 (no final activation)."""

    def __init__(self, input_dim: int, dropout: float = 0.3):
        super().__init__()
        self.input_dim  = input_dim
        self.output_dim = 64
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, 128),       nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)


# ─── Geo Risk Feature Extractor ───────────────────────────────────────────────
class GeoRiskFeatureExtractor(nn.Module):
    """GeoNet: 1 → 32 → 64 with ReLU."""

    def __init__(self):
        super().__init__()
        self.input_dim  = 1
        self.output_dim = 64
        self.network = nn.Sequential(
            nn.Linear(1, 32),  nn.ReLU(inplace=True),
            nn.Linear(32, 64), nn.ReLU(inplace=True),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)


# ─── Multimodal Transformer ───────────────────────────────────────────────────
class MultiModalTransformer(nn.Module):
    """Projection → Transformer Encoder → Flatten → Classifier (logit out)."""

    def __init__(self, image_dim=1280, clinical_dim=64, geo_dim=64,
                 latent_dim=128, num_heads=4, num_layers=2,
                 ffn_dim=256, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.fused_dim  = latent_dim * 3

        self.image_projection    = nn.Linear(image_dim,    latent_dim)
        self.clinical_projection = nn.Linear(clinical_dim, latent_dim)
        self.geo_projection      = nn.Linear(geo_dim,      latent_dim)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=latent_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, activation="relu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.Linear(self.fused_dim, 256), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 64),             nn.ReLU(inplace=True),
            nn.Linear(64, 1),               # raw logit — sigmoid applied at eval
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, image_features, clinical_features, geo_features):
        img_tok  = self.image_projection(image_features)
        clin_tok = self.clinical_projection(clinical_features)
        geo_tok  = self.geo_projection(geo_features)
        tokens       = torch.stack([img_tok, clin_tok, geo_tok], dim=1)
        fused_tokens = self.transformer(tokens)
        fused_vector = fused_tokens.flatten(start_dim=1)
        return self.classifier(fused_vector)


# ─── Wrapper that Phase 5 uses ────────────────────────────────────────────────
class MultimodalAnemiaModel(nn.Module):
    """
    Wraps all four Phase-3/4 components into one module.
    Matches Phase 5's MultimodalAnemiaModel exactly so state_dict keys align.
    """

    def __init__(self, image_ext, clinical_ext, geo_ext, transformer):
        super().__init__()
        self.image_ext    = image_ext
        self.clinical_ext = clinical_ext
        self.geo_ext      = geo_ext
        self.transformer  = transformer

    def freeze_backbone(self):
        for ext in [self.image_ext, self.clinical_ext, self.geo_ext]:
            for p in ext.parameters():
                p.requires_grad = False

    def unfreeze_backbone(self, mode: str = "partial"):
        if mode == "full":
            for ext in [self.image_ext, self.clinical_ext, self.geo_ext]:
                for p in ext.parameters():
                    p.requires_grad = True
        else:  # partial — last 2 EfficientNet blocks + all clinical/geo
            if hasattr(self.image_ext, "backbone"):
                blocks = list(self.image_ext.backbone.children())
                for block in blocks[-2:]:
                    for p in block.parameters():
                        p.requires_grad = True
            for ext in [self.clinical_ext, self.geo_ext]:
                for p in ext.parameters():
                    p.requires_grad = True

    def trainable_params(self):
        return [p for p in self.parameters() if p.requires_grad]

    def forward(self, images, clinical, geo):
        img_feat  = self.image_ext(images)
        clin_feat = self.clinical_ext(clinical)
        geo_feat  = self.geo_ext(geo)
        return self.transformer(img_feat, clin_feat, geo_feat)


print("Architecture classes re-declared (identical to Phase 5).")


Architecture classes re-declared (identical to Phase 5).


In [5]:
# =============================================================================
#  2.2  Checkpoint utility helpers — same as Phase 5
# =============================================================================

def _load_raw(path: Path, label: str) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"[{label}] Checkpoint not found: {path}")
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        raise RuntimeError(f"[{label}] Failed to deserialise {path}: {e}") from e


def _meta(ckpt: dict, label: str) -> dict:
    """Return the architecture dict, accepting both 'metadata' and 'config' keys."""
    for key in ("metadata", "config"):
        if key in ckpt:
            return ckpt[key]
    raise KeyError(
        f"[{label}] Checkpoint has neither 'metadata' nor 'config' key. "
        f"Found keys: {list(ckpt.keys())}"
    )


def _load_state(model: nn.Module, ckpt: dict, label: str) -> None:
    sd = ckpt.get("state_dict", ckpt)
    try:
        model.load_state_dict(sd, strict=True)
    except RuntimeError as e:
        raise RuntimeError(
            f"[{label}] state_dict mismatch — did you regenerate Phase 3/4 "
            f"after changing the architecture?\n  Original error: {e}"
        ) from e


def _check_no_nans(model: nn.Module, name: str) -> None:
    bad = [(n, p) for n, p in model.named_parameters() if torch.isnan(p).any()]
    if bad:
        details = "\n".join(f"  {n}: {torch.isnan(p).sum().item()} NaN(s)" for n, p in bad)
        raise RuntimeError(
            f"NaN weights detected in {name} after loading checkpoint:\n{details}"
        )
    print(f"  NaN check passed ✓  ({name})")


print("Checkpoint utilities ready.")


Checkpoint utilities ready.


In [6]:
# =============================================================================
#  2.3  Load processed data from Phase 2 (same logic as Phase 5 Section 3)
# =============================================================================
CLINICAL_CSV = PROCESSED_DIR / "clinical_processed.csv"
GEO_CSV      = PROCESSED_DIR / "geo_processed.csv"

REQUIRED_CLINICAL_DIM = EVAL_CONFIG["required_clinical_dim"]

LABEL_COL = "Label"
_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def _clean_df(df: pd.DataFrame, extra_drop=()) -> pd.DataFrame:
    drop = [c for c in df.columns
            if c.startswith("Unnamed:") or c in ("Patient_ID", "Note") or c in extra_drop]
    return df.drop(columns=drop)

# ── Clinical ──────────────────────────────────────────────────────────────────
if CLINICAL_CSV.exists():
    clinical_df_raw = pd.read_csv(CLINICAL_CSV)
    clinical_df_raw = _clean_df(clinical_df_raw)
    HAS_CLINICAL    = True
    print(f"Clinical CSV  : {CLINICAL_CSV}  {clinical_df_raw.shape}")
else:
    clinical_df_raw = None
    HAS_CLINICAL    = False
    print("WARNING: clinical_processed.csv not found — synthetic data will be used.")

# ── Geo ───────────────────────────────────────────────────────────────────────
if GEO_CSV.exists():
    geo_df_raw = pd.read_csv(GEO_CSV)
    geo_df_raw = _clean_df(geo_df_raw)
    HAS_GEO    = True
    print(f"Geo CSV       : {GEO_CSV}  {geo_df_raw.shape}")
else:
    geo_df_raw = None
    HAS_GEO    = False
    print("WARNING: geo_processed.csv not found — fallback State_Risk = 0.534.")


Clinical CSV  : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\dataset\processed\clinical_processed.csv  (95, 5)
Geo CSV       : C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\dataset\processed\geo_processed.csv  (95, 4)


In [7]:
# =============================================================================
#  2.4  Resolve labels, features and geo risk (mirrors Phase 5 Section 3.2/3.3)
# =============================================================================
if HAS_CLINICAL:
    # Derive Label from Hgb if absent
    if LABEL_COL not in clinical_df_raw.columns:
        if "Hgb" not in clinical_df_raw.columns:
            raise RuntimeError("Cannot create Label: Hgb column is missing.")
        clinical_df_raw[LABEL_COL] = (clinical_df_raw["Hgb"] < 12).astype(int)

    # Required feature columns — must match what Phase 3 was trained with
    # ------------------------------------------------------------------
    # Clinical feature columns (must match Phase 3 checkpoint exactly)
    # ------------------------------------------------------------------

    expected_cols = [
        "Hgb",
        "Age",
        "Gender_F",
        "Gender_M"
    ]

    missing = [c for c in expected_cols if c not in clinical_df_raw.columns]

    if missing:
        raise RuntimeError(
            f"Missing clinical feature columns: {missing}\n"
            f"Available columns: {clinical_df_raw.columns.tolist()}"
        )

    feature_cols = expected_cols

    clinical_features_df = clinical_df_raw[feature_cols].copy()
    labels               = clinical_df_raw[LABEL_COL].values.astype(np.float32)
    clinical_input_dim   = clinical_features_df.shape[1]

    print(f"Feature columns ({clinical_input_dim}): {feature_cols}")
    print(f"Label distribution: {int((labels==1).sum())} anemic / "
          f"{int((labels==0).sum())} non-anemic")
else:
    # Synthetic fallback (matches Phase 5 behaviour)
    rng = np.random.default_rng(SEED)
    N   = 200
    clinical_features_df = pd.DataFrame(rng.normal(0, 1, (N, REQUIRED_CLINICAL_DIM)),
                                         columns=[f"feat_{i}" for i in range(REQUIRED_CLINICAL_DIM)])
    labels               = rng.integers(0, 2, N).astype(np.float32)
    clinical_input_dim   = REQUIRED_CLINICAL_DIM
    print(f"Using synthetic data: N={N}, clinical_dim={clinical_input_dim}")

# ── Geo risk ──────────────────────────────────────────────────────────────────
if HAS_GEO and "State_Risk" in geo_df_raw.columns:
    if len(geo_df_raw) == len(clinical_features_df):
        state_risk = geo_df_raw["State_Risk"].values.astype(np.float32)
    else:
        state_risk = np.full(len(clinical_features_df),
                             float(geo_df_raw["State_Risk"].iloc[0]),
                             dtype=np.float32)
else:
    state_risk = np.full(len(clinical_features_df), 0.534, dtype=np.float32)

print(f"State_Risk range : [{state_risk.min():.3f}, {state_risk.max():.3f}]")
print(f"Total samples    : {len(clinical_features_df)}")


Feature columns (4): ['Hgb', 'Age', 'Gender_F', 'Gender_M']
Label distribution: 56 anemic / 39 non-anemic
State_Risk range : [0.534, 0.534]
Total samples    : 95


In [8]:
# =============================================================================
#  2.5  AnemiaDataset class — identical to Phase 5 so the same splits apply
# =============================================================================
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])


class AnemiaDataset(Dataset):
    """Returns (image_tensor, clinical_tensor, geo_tensor, label) per sample."""

    def __init__(self, clinical_arr, geo_arr, label_arr, image_dir, transform=None):
        self.clinical  = clinical_arr.astype(np.float32)
        self.geo       = geo_arr.astype(np.float32).reshape(-1, 1)
        self.labels    = label_arr.astype(np.float32)
        self.image_dir = image_dir
        self.transform = transform or eval_transform

        self.class_images = {0: [], 1: []}
        if image_dir.exists():
            for cls, folder in [(0, "non_anemic"), (1, "anemic")]:
                p = image_dir / folder
                if p.exists():
                    self.class_images[cls] = sorted(p.glob("*.*"))
        self.has_images = any(len(v) > 0 for v in self.class_images.values())

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = int(self.labels[idx])

        if self.has_images and len(self.class_images.get(label, [])) > 0:
            from PIL import Image
            paths = self.class_images[label]
            img   = Image.open(paths[idx % len(paths)]).convert("RGB")
            image_tensor = self.transform(img)
        else:
            g = torch.Generator().manual_seed(idx)
            image_tensor = torch.randn(3, 224, 224, generator=g)

        return (
            image_tensor,
            torch.tensor(self.clinical[idx], dtype=torch.float32),
            torch.tensor(self.geo[idx],      dtype=torch.float32),
            torch.tensor(self.labels[idx],   dtype=torch.float32),
        )


print("AnemiaDataset ready.")


AnemiaDataset ready.


In [9]:
# =============================================================================
#  2.6  Reproduce the exact same 70/15/15 split as Phase 5
#       (same SEED → same train/val/test indices → fair comparison)
# =============================================================================
_clin_arr = clinical_features_df.values.astype(np.float32)
_geo_arr  = state_risk
_lab_arr  = labels

_idx       = np.arange(len(_lab_arr))
train_idx, temp_idx = train_test_split(
    _idx, test_size=0.30, random_state=SEED, stratify=_lab_arr
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED, stratify=_lab_arr[temp_idx]
)

print(f"Split — Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

# We only need val + test for evaluation; train is kept for class-imbalance reference
val_dataset  = AnemiaDataset(_clin_arr[val_idx],  _geo_arr[val_idx],  _lab_arr[val_idx],  IMAGE_DIR)
test_dataset = AnemiaDataset(_clin_arr[test_idx], _geo_arr[test_idx], _lab_arr[test_idx], IMAGE_DIR)

val_loader  = DataLoader(val_dataset,  batch_size=EVAL_CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=EVAL_CONFIG["batch_size"], shuffle=False)

print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")
print(f"Images found on disk : {test_dataset.has_images}")


Split — Train: 66  Val: 14  Test: 15
Val batches  : 1
Test batches : 1
Images found on disk : False


In [10]:
# =============================================================================
#  2.7  Load the four Phase 3/4 component checkpoints and assemble full model
# =============================================================================
IMAGE_CKPT_PATH    = MODELS_DIR / "image_feature_extractor.pth"
CLINICAL_CKPT_PATH = MODELS_DIR / "clinical_feature_extractor.pth"
GEO_CKPT_PATH      = MODELS_DIR / "geo_feature_extractor.pth"
TRANSFORMER_PATH   = MODELS_DIR / "multimodal_transformer.pth"

for _p in [IMAGE_CKPT_PATH, CLINICAL_CKPT_PATH, GEO_CKPT_PATH, TRANSFORMER_PATH]:
    status = "FOUND  " if _p.exists() else "MISSING"
    print(f"  {status} {_p.name}")

# ── Image extractor ───────────────────────────────────────────────────────────
_ckpt_img  = _load_raw(IMAGE_CKPT_PATH, "Image")
_meta_img  = _meta(_ckpt_img, "Image")
image_extractor = ImageFeatureExtractor(
    backbone_name=_meta_img.get("backbone", "EfficientNet-B0"), pretrained=False
).to(DEVICE)
_load_state(image_extractor, _ckpt_img, "Image")
_check_no_nans(image_extractor, "ImageFeatureExtractor")

# ── Clinical extractor ────────────────────────────────────────────────────────
_ckpt_clin = _load_raw(CLINICAL_CKPT_PATH, "Clinical")
_meta_clin = _meta(_ckpt_clin, "Clinical")
_ckpt_dim  = int(_meta_clin.get("input_dim", clinical_input_dim))

if _ckpt_dim != clinical_input_dim:
    raise RuntimeError(
        f"Clinical checkpoint expects {_ckpt_dim} features but dataset has "
        f"{clinical_input_dim}.\n"
        f"Regenerate Phase 3 with the current dataset schema to fix this."
    )

clinical_extractor = ClinicalFeatureExtractor(input_dim=clinical_input_dim).to(DEVICE)
_load_state(clinical_extractor, _ckpt_clin, "Clinical")
_check_no_nans(clinical_extractor, "ClinicalFeatureExtractor")

# ── Geo extractor ─────────────────────────────────────────────────────────────
_ckpt_geo = _load_raw(GEO_CKPT_PATH, "Geo")
geo_extractor = GeoRiskFeatureExtractor().to(DEVICE)
_load_state(geo_extractor, _ckpt_geo, "Geo")
_check_no_nans(geo_extractor, "GeoRiskFeatureExtractor")

# ── Transformer / fusion ──────────────────────────────────────────────────────
_ckpt_trans = _load_raw(TRANSFORMER_PATH, "Transformer")
_cfg_trans  = _meta(_ckpt_trans, "Transformer")
transformer = MultiModalTransformer(
    image_dim    = _cfg_trans.get("image_dim",    1280),
    clinical_dim = _cfg_trans.get("clinical_dim", 64),
    geo_dim      = _cfg_trans.get("geo_dim",      64),
    latent_dim   = _cfg_trans.get("latent_dim",   128),
    num_heads    = _cfg_trans.get("num_heads",    4),
    num_layers   = _cfg_trans.get("num_layers",   2),
    ffn_dim      = _cfg_trans.get("ffn_dim",      256),
    dropout      = _cfg_trans.get("dropout",      0.1),
).to(DEVICE)
_load_state(transformer, _ckpt_trans, "Transformer")
_check_no_nans(transformer, "MultiModalTransformer")

# ── Assemble wrapper ──────────────────────────────────────────────────────────
model = MultimodalAnemiaModel(
    image_ext=image_extractor,
    clinical_ext=clinical_extractor,
    geo_ext=geo_extractor,
    transformer=transformer,
).to(DEVICE)

print()
print("Components loaded. Now loading best end-to-end checkpoint from Phase 5...")


  FOUND   image_feature_extractor.pth
  FOUND   clinical_feature_extractor.pth
  FOUND   geo_feature_extractor.pth
  FOUND   multimodal_transformer.pth
  NaN check passed ✓  (ImageFeatureExtractor)
  NaN check passed ✓  (ClinicalFeatureExtractor)
  NaN check passed ✓  (GeoRiskFeatureExtractor)
  NaN check passed ✓  (MultiModalTransformer)

Components loaded. Now loading best end-to-end checkpoint from Phase 5...


In [11]:
# =============================================================================
#  2.8  Load the best end-to-end checkpoint saved by Phase 5 Stage 3
# =============================================================================
BEST_CKPT_PATH = MODELS_DIR / EVAL_CONFIG["best_ckpt"]

if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: {BEST_CKPT_PATH}\n"
        f"Run Phase 5 (Training Strategy) first to generate this file."
    )

_best_ckpt = torch.load(
    BEST_CKPT_PATH,
    map_location=DEVICE,
    weights_only=False
)

state_dict = _best_ckpt["state_dict"]

# Automatically support both naming conventions
rename_map = {
    "transformer.image_proj.": "transformer.image_projection.",
    "transformer.clinical_proj.": "transformer.clinical_projection.",
    "transformer.geo_proj.": "transformer.geo_projection.",
}

new_state = {}

for key, value in state_dict.items():
    new_key = key
    for old, new in rename_map.items():
        if key.startswith(old):
            new_key = key.replace(old, new, 1)
            break
    new_state[new_key] = value

missing, unexpected = model.load_state_dict(new_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
model.eval()

_saved_metrics = _best_ckpt.get("metrics", {})
_saved_stage   = _best_ckpt.get("metadata", {}).get("stage", "?")
_saved_epoch   = _best_ckpt.get("epoch", "?")

print(f"Loaded: {BEST_CKPT_PATH.name}")
print(f"  Saved at  : Stage {_saved_stage}, Epoch {_saved_epoch}")
if _saved_metrics:
    print(f"  Val F1    : {_saved_metrics.get('f1', '?'):.4f}")
    print(f"  Val AUC   : {_saved_metrics.get('roc_auc', '?'):.4f}")
print()

# Compute pos_weight from training labels (used in loss for baselines too)
_train_y = _lab_arr[train_idx]
_pos     = int(_train_y.sum())
_neg     = int(len(_train_y) - _pos)
pos_weight = (
    torch.tensor([_neg / _pos], dtype=torch.float32).to(DEVICE)
    if _pos > 0 and _neg > 0
    else torch.ones(1, device=DEVICE)
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"pos_weight = {pos_weight.item():.4f}  (neg={_neg}, pos={_pos})")
print("\nPhase 5 best model loaded and ready for evaluation.")


Missing keys: []
Unexpected keys: []
Loaded: best_multimodal_model.pth
  Saved at  : Stage 3, Epoch 1
  Val F1    : 0.8571
  Val AUC   : 0.8542

pos_weight = 0.6923  (neg=27, pos=39)

Phase 5 best model loaded and ready for evaluation.


---
## Section 3 — Full Model Evaluation on Test Set

Runs a complete forward pass over the held-out test set and reports the
full metric suite: Accuracy, Precision, Recall, F1, ROC-AUC, Specificity,
Sensitivity, MCC, and Brier Score.


In [12]:
# =============================================================================
#  3.1  Evaluation loop — mirrors Phase 5's _eval_loop
# =============================================================================

def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module,
             split_name: str = "Test") -> tuple:
    """
    Run a full evaluation pass.
    Returns: (avg_loss, y_true ndarray, y_prob ndarray)
    """
    model.eval()
    losses, y_true, y_prob = [], [], []

    with torch.no_grad():
        for images, clinical, geo, lbl in tqdm(loader, desc=f"  [{split_name}]", leave=False):
            images   = images.to(DEVICE)
            clinical = clinical.to(DEVICE)
            geo      = geo.to(DEVICE)
            lbl      = lbl.to(DEVICE).unsqueeze(1)

            logits = model(images, clinical, geo)
            loss   = criterion(logits, lbl.float())

            if torch.isnan(loss):
                raise RuntimeError(f"NaN loss during {split_name} evaluation.")

            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            losses.append(loss.item())
            y_prob.extend(probs)
            y_true.extend(lbl.cpu().numpy().ravel())

    return float(np.mean(losses)), np.array(y_true), np.array(y_prob)


# =============================================================================
#  3.2  Full metric helper
# =============================================================================
def full_metrics(y_true: np.ndarray, y_prob: np.ndarray,
                 threshold: float = 0.5) -> dict:
    """Return the complete evaluation metric suite."""
    y_pred = (y_prob >= threshold).astype(int)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = float("nan")

    try:
        brier = brier_score_loss(y_true, y_prob)
    except Exception:
        brier = float("nan")

    try:
        mcc = matthews_corrcoef(y_true, y_pred)
    except Exception:
        mcc = float("nan")

    # Confusion matrix elements — guard against single-class splits
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except ValueError:
        tn, fp, fn, tp = 0, 0, 0, int(y_true.sum())

    return {
        "accuracy"   : accuracy_score(y_true, y_pred),
        "precision"  : precision_score(y_true, y_pred, zero_division=0),
        "recall"     : recall_score(y_true, y_pred, zero_division=0),
        "f1"         : f1_score(y_true, y_pred, zero_division=0),
        "roc_auc"    : auc,
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "sensitivity": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        "mcc"        : mcc,
        "brier"      : brier,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


print("evaluate() and full_metrics() ready.")


evaluate() and full_metrics() ready.


In [13]:
# =============================================================================
#  3.3  Run full model on test set
# =============================================================================
test_loss, y_test_true, y_test_prob = evaluate(model, test_loader, criterion, "Full model")
y_test_pred = (y_test_prob >= EVAL_CONFIG["threshold"]).astype(int)

if len(y_test_true) == 0:
    raise RuntimeError("Test set is empty — check DataLoader and dataset split.")

full_model_metrics = full_metrics(y_test_true, y_test_prob, EVAL_CONFIG["threshold"])

print("=" * 58)
print("  FULL MODEL — TEST SET RESULTS")
print("=" * 58)
print(f"  Samples        : {len(y_test_true)}")
print(f"  Positives      : {int(y_test_true.sum())}  "
      f"({100*y_test_true.mean():.1f}%)")
print(f"  Test Loss      : {test_loss:.4f}")
print()
print(f"  Accuracy       : {full_model_metrics['accuracy']:.4f}")
print(f"  Precision      : {full_model_metrics['precision']:.4f}")
print(f"  Recall         : {full_model_metrics['recall']:.4f}")
print(f"  Specificity    : {full_model_metrics['specificity']:.4f}")
print(f"  Sensitivity    : {full_model_metrics['sensitivity']:.4f}")
print(f"  F1 Score       : {full_model_metrics['f1']:.4f}")
print(f"  ROC-AUC        : {full_model_metrics['roc_auc']:.4f}")
print(f"  MCC            : {full_model_metrics['mcc']:.4f}")
print(f"  Brier Score    : {full_model_metrics['brier']:.4f}")
print("=" * 58)


  [Full model]:   0%|          | 0/1 [00:00<?, ?it/s]

  FULL MODEL — TEST SET RESULTS
  Samples        : 15
  Positives      : 9  (60.0%)
  Test Loss      : 0.3941

  Accuracy       : 0.7333
  Precision      : 1.0000
  Recall         : 0.5556
  Specificity    : 1.0000
  Sensitivity    : 0.5556
  F1 Score       : 0.7143
  ROC-AUC        : 0.9444
  MCC            : 0.5774
  Brier Score    : 0.1797


---
## Section 4 — Ablation Study: Modality Contribution Baselines

Three ablation variants are evaluated against the full model to quantify
how much each modality contributes:

| Variant | Description |
|---------|-------------|
| **Image-only** | EfficientNet features → small classifier; clinical/geo inputs are zeroed |
| **Clinical-only** | MLP features → small classifier; image/geo inputs are zeroed |
| **No-Geo** | Full model with geo embedding zeroed (geo extractor bypassed) |

All variants reuse the **same trained weights** from the best checkpoint —
no retraining is performed. Zeroing a modality's input gives the model the
same tensor shape it expects but removes the information content from that
stream, isolating each modality's marginal contribution.


In [14]:
# =============================================================================
#  4.1  Zeroing-based ablation evaluation
#
#  Design rationale: replacing a modality's input with zeros preserves
#  the tensor shapes that the projections/transformer expect, avoids any
#  data-leakage from the held-out set, and is computationally free.
# =============================================================================

def evaluate_ablation(model: nn.Module, loader: DataLoader,
                      criterion: nn.Module,
                      zero_image: bool = False,
                      zero_clinical: bool = False,
                      zero_geo: bool = False,
                      variant_name: str = "") -> tuple:
    """
    Evaluate the full model with one or more input modalities zeroed out.

    Args:
        zero_image    : Replace image input with an all-zero tensor.
        zero_clinical : Replace clinical input with an all-zero tensor.
        zero_geo      : Replace geo input with an all-zero tensor.
        variant_name  : Label for tqdm progress bar.

    Returns: (avg_loss, y_true, y_prob)
    """
    model.eval()
    losses, y_true, y_prob = [], [], []

    desc = f"  [{variant_name}]"
    with torch.no_grad():
        for images, clinical, geo, lbl in tqdm(loader, desc=desc, leave=False):
            images   = images.to(DEVICE)
            clinical = clinical.to(DEVICE)
            geo      = geo.to(DEVICE)
            lbl      = lbl.to(DEVICE).unsqueeze(1)

            if zero_image:
                images   = torch.zeros_like(images)
            if zero_clinical:
                clinical = torch.zeros_like(clinical)
            if zero_geo:
                geo      = torch.zeros_like(geo)

            logits = model(images, clinical, geo)
            loss   = criterion(logits, lbl.float())

            if torch.isnan(loss):
                raise RuntimeError(f"NaN loss in ablation: {variant_name}")

            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            losses.append(loss.item())
            y_prob.extend(probs)
            y_true.extend(lbl.cpu().numpy().ravel())

    return float(np.mean(losses)), np.array(y_true), np.array(y_prob)


print("evaluate_ablation() ready.")


evaluate_ablation() ready.


In [15]:
# =============================================================================
#  4.2  Run all three ablation baselines
# =============================================================================
print("Running ablation baselines (same test set, zeroed inputs)...")
print()

# ── Baseline 1: Image-only (zero clinical + geo) ──────────────────────────────
_, y_img_true, y_img_prob = evaluate_ablation(
    model, test_loader, criterion,
    zero_clinical=True, zero_geo=True,
    variant_name="Image-only"
)
img_only_metrics = full_metrics(y_img_true, y_img_prob, EVAL_CONFIG["threshold"])
print(f"Image-only     — F1={img_only_metrics['f1']:.4f}  "
      f"AUC={img_only_metrics['roc_auc']:.4f}  "
      f"Acc={img_only_metrics['accuracy']:.4f}")

# ── Baseline 2: Clinical-only (zero image + geo) ──────────────────────────────
_, y_clin_true, y_clin_prob = evaluate_ablation(
    model, test_loader, criterion,
    zero_image=True, zero_geo=True,
    variant_name="Clinical-only"
)
clin_only_metrics = full_metrics(y_clin_true, y_clin_prob, EVAL_CONFIG["threshold"])
print(f"Clinical-only  — F1={clin_only_metrics['f1']:.4f}  "
      f"AUC={clin_only_metrics['roc_auc']:.4f}  "
      f"Acc={clin_only_metrics['accuracy']:.4f}")

# ── Baseline 3: No-Geo (zero geo only; image + clinical active) ───────────────
_, y_nogeo_true, y_nogeo_prob = evaluate_ablation(
    model, test_loader, criterion,
    zero_geo=True,
    variant_name="No-Geo"
)
no_geo_metrics = full_metrics(y_nogeo_true, y_nogeo_prob, EVAL_CONFIG["threshold"])
print(f"No-Geo         — F1={no_geo_metrics['f1']:.4f}  "
      f"AUC={no_geo_metrics['roc_auc']:.4f}  "
      f"Acc={no_geo_metrics['accuracy']:.4f}")

print()
print(f"Full Model     — F1={full_model_metrics['f1']:.4f}  "
      f"AUC={full_model_metrics['roc_auc']:.4f}  "
      f"Acc={full_model_metrics['accuracy']:.4f}")


Running ablation baselines (same test set, zeroed inputs)...



  [Image-only]:   0%|          | 0/1 [00:00<?, ?it/s]

Image-only     — F1=0.5333  AUC=0.5556  Acc=0.5333


  [Clinical-only]:   0%|          | 0/1 [00:00<?, ?it/s]

Clinical-only  — F1=0.6957  AUC=0.8704  Acc=0.5333


  [No-Geo]:   0%|          | 0/1 [00:00<?, ?it/s]

No-Geo         — F1=0.6154  AUC=0.9444  Acc=0.6667

Full Model     — F1=0.7143  AUC=0.9444  Acc=0.7333


---
## Section 5 — Modality Contribution Comparison


In [16]:
# =============================================================================
#  5.1  Build comparison dataframe
# =============================================================================
VARIANTS = {
    "Image-only"  : img_only_metrics,
    "Clinical-only": clin_only_metrics,
    "No-Geo"      : no_geo_metrics,
    "Full Model"  : full_model_metrics,
}

METRICS_DISPLAY = ["accuracy", "precision", "recall", "f1", "roc_auc",
                   "specificity", "sensitivity", "mcc"]

comparison_df = pd.DataFrame(
    {name: {m: mets[m] for m in METRICS_DISPLAY} for name, mets in VARIANTS.items()}
).T.round(4)

comparison_df.index.name = "Variant"

print("=" * 75)
print("  MODALITY CONTRIBUTION — COMPARISON TABLE")
print("=" * 75)
print(comparison_df.to_string())
print()

# Save to CSV
_csv_path = OUTPUTS_DIR / "ablation_comparison.csv"
comparison_df.to_csv(_csv_path)
print(f"Saved -> {_csv_path}")


  MODALITY CONTRIBUTION — COMPARISON TABLE
               accuracy  precision  recall      f1  roc_auc  specificity  sensitivity     mcc
Variant                                                                                      
Image-only       0.5333     0.6667  0.4444  0.5333   0.5556       0.6667       0.4444  0.1111
Clinical-only    0.5333     0.5714  0.8889  0.6957   0.8704       0.0000       0.8889 -0.2182
No-Geo           0.6667     1.0000  0.4444  0.6154   0.9444       1.0000       0.4444  0.4924
Full Model       0.7333     1.0000  0.5556  0.7143   0.9444       1.0000       0.5556  0.5774

Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\ablation_comparison.csv


In [17]:
# =============================================================================
#  5.2  Bar chart — all key metrics side-by-side across all four variants
# =============================================================================
_plot_metrics = ["accuracy", "f1", "roc_auc", "precision", "recall", "specificity"]
_metric_labels = ["Accuracy", "F1", "ROC-AUC", "Precision", "Recall", "Specificity"]

_colors = {
    "Image-only"   : "#5DA5DA",
    "Clinical-only": "#FAA43A",
    "No-Geo"       : "#60BD68",
    "Full Model"   : "#F15854",
}

n_metrics  = len(_plot_metrics)
n_variants = len(VARIANTS)
x          = np.arange(n_metrics)
width      = 0.18
offsets    = np.linspace(-(n_variants-1)/2, (n_variants-1)/2, n_variants) * width

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("AnemiaFusionNet — Modality Contribution Ablation Study",
             fontsize=13, fontweight="bold")

for (name, mets), offset in zip(VARIANTS.items(), offsets):
    vals = [mets[m] for m in _plot_metrics]
    bars = ax.bar(x + offset, vals, width, label=name,
                  color=_colors[name], edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=7, color="#333")

ax.set_xticks(x)
ax.set_xticklabels(_metric_labels, fontsize=10)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.12)
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
_p = OUTPUTS_DIR / "ablation_comparison_bar.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\ablation_comparison_bar.png


---
## Section 6 — Confusion Matrix

In [18]:
# =============================================================================
#  6.1  Professional confusion matrix heatmap — full model vs baselines
# =============================================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Confusion Matrices — Full Model vs Ablation Baselines",
             fontsize=13, fontweight="bold")

_cm_variants = [
    ("Image-only",    y_img_true,   (y_img_prob   >= EVAL_CONFIG["threshold"]).astype(int)),
    ("Clinical-only", y_clin_true,  (y_clin_prob  >= EVAL_CONFIG["threshold"]).astype(int)),
    ("No-Geo",        y_nogeo_true, (y_nogeo_prob >= EVAL_CONFIG["threshold"]).astype(int)),
    ("Full Model",    y_test_true,  y_test_pred),
]

for ax, (title, y_t, y_p) in zip(axes, _cm_variants):
    cm = confusion_matrix(y_t, y_p, labels=[0, 1])
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-Anemic", "Anemic"], fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Non-Anemic", "Anemic"], fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(title, fontweight="bold", fontsize=11)

    thresh = cm.max() / 2
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    fontsize=14, fontweight="bold",
                    color="white" if cm[i, j] > thresh else "black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
_p = OUTPUTS_DIR / "confusion_matrix_all.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\confusion_matrix_all.png


---
## Section 7 — ROC Curve (Full Model + All Baselines)

In [19]:
# =============================================================================
#  7.1  ROC curves — one line per variant
# =============================================================================
_roc_variants = [
    ("Image-only",    y_img_true,    y_img_prob,    "#5DA5DA", "--"),
    ("Clinical-only", y_clin_true,   y_clin_prob,   "#FAA43A", "-."),
    ("No-Geo",        y_nogeo_true,  y_nogeo_prob,  "#60BD68", ":"),
    ("Full Model",    y_test_true,   y_test_prob,   "#F15854", "-"),
]

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot([0, 1], [0, 1], color="#999", linestyle="--", linewidth=1.2,
        label="Random classifier")

for name, y_t, y_p, color, ls in _roc_variants:
    try:
        fpr, tpr, _ = roc_curve(y_t, y_p)
        auc          = roc_auc_score(y_t, y_p)
        ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=2.5,
                label=f"{name}  (AUC = {auc:.3f})")
    except ValueError as e:
        print(f"  Skipping ROC for {name}: {e}")

ax.fill_between(*roc_curve(y_test_true, y_test_prob)[:2],
                alpha=0.06, color="#F15854")
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
ax.set_title("ROC Curves — AnemiaFusionNet & Ablation Baselines",
             fontsize=12, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
_p = OUTPUTS_DIR / "roc_curve_all.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\roc_curve_all.png


---
## Section 8 — Precision-Recall Curve (Full Model + All Baselines)

In [20]:
# =============================================================================
#  8.1  Precision-Recall curves — all variants overlaid
# =============================================================================
_pr_variants = [
    ("Image-only",    y_img_true,    y_img_prob,    "#5DA5DA", "--"),
    ("Clinical-only", y_clin_true,   y_clin_prob,   "#FAA43A", "-."),
    ("No-Geo",        y_nogeo_true,  y_nogeo_prob,  "#60BD68", ":"),
    ("Full Model",    y_test_true,   y_test_prob,   "#F15854", "-"),
]

baseline_prev = y_test_true.sum() / len(y_test_true)

fig, ax = plt.subplots(figsize=(8, 7))
ax.axhline(baseline_prev, color="#999", linestyle="--", linewidth=1.2,
           label=f"No-skill (prevalence = {baseline_prev:.3f})")

for name, y_t, y_p, color, ls in _pr_variants:
    try:
        prec_vals, rec_vals, _ = precision_recall_curve(y_t, y_p)
        f1_val = full_metrics(y_t, y_p)["f1"]
        ax.plot(rec_vals, prec_vals, color=color, linestyle=ls, linewidth=2.5,
                label=f"{name}  (F1 = {f1_val:.3f})")
    except Exception as e:
        print(f"  Skipping PR for {name}: {e}")

ax.fill_between(*precision_recall_curve(y_test_true, y_test_prob)[1::-1],
                alpha=0.06, color="#F15854")
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision-Recall Curves — AnemiaFusionNet & Ablation Baselines",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
_p = OUTPUTS_DIR / "pr_curve_all.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\pr_curve_all.png


---
## Section 9 — Classification Report (Full Model)

In [21]:
# =============================================================================
#  9.1  Full classification report
# =============================================================================
report_str = classification_report(
    y_test_true, y_test_pred,
    labels=[0, 1],
    target_names=["Non-Anemic", "Anemic"],
    zero_division=0,
)
print("Classification Report — Full Model (Test Set)")
print("=" * 55)
print(report_str)

# Save structured CSV
report_df = pd.DataFrame(
    classification_report(
        y_test_true, y_test_pred,
        labels=[0, 1],
        target_names=["Non-Anemic", "Anemic"],
        output_dict=True, zero_division=0,
    )
).transpose()

_p = OUTPUTS_DIR / "classification_report.csv"
report_df.to_csv(_p)
print(f"Saved -> {_p}")


Classification Report — Full Model (Test Set)
              precision    recall  f1-score   support

  Non-Anemic       0.60      1.00      0.75         6
      Anemic       1.00      0.56      0.71         9

    accuracy                           0.73        15
   macro avg       0.80      0.78      0.73        15
weighted avg       0.84      0.73      0.73        15

Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\classification_report.csv


---
## Section 10 — Calibration Analysis (Reliability Diagram)

A well-calibrated classifier produces predicted probabilities that match
the observed positive rate. The diagonal line represents perfect calibration.
Points above the diagonal indicate under-confidence; points below indicate
over-confidence.


In [22]:
# =============================================================================
#  10.1  Reliability diagram (calibration curve)
# =============================================================================
n_bins = EVAL_CONFIG["calibration_bins"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Probability Calibration — AnemiaFusionNet (Full Model)",
             fontsize=12, fontweight="bold")

# ── Reliability diagram ───────────────────────────────────────────────────────
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Perfect calibration")

try:
    prob_true, prob_pred = calibration_curve(
        y_test_true, y_test_prob, n_bins=n_bins, strategy="uniform"
    )
    brier = brier_score_loss(y_test_true, y_test_prob)
    ax.plot(prob_pred, prob_true, "o-", color="#F15854", linewidth=2, markersize=6,
            label=f"Full Model  (Brier = {brier:.4f})")
    ax.fill_between(prob_pred, prob_pred, prob_true,
                    alpha=0.08, color="#F15854", label="Calibration gap")
except Exception as e:
    print(f"  Calibration curve failed: {e}")

ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Reliability Diagram")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)

# ── Predicted probability histogram ──────────────────────────────────────────
ax2 = axes[1]
ax2.hist(y_test_prob[y_test_true == 0], bins=20, alpha=0.6, color="#5DA5DA",
         label="Non-Anemic (true)", edgecolor="white")
ax2.hist(y_test_prob[y_test_true == 1], bins=20, alpha=0.6, color="#F15854",
         label="Anemic (true)", edgecolor="white")
ax2.axvline(EVAL_CONFIG["threshold"], color="black", linestyle="--", linewidth=1.5,
            label=f"Threshold = {EVAL_CONFIG['threshold']}")
ax2.set_xlabel("Predicted Probability"); ax2.set_ylabel("Count")
ax2.set_title("Probability Distribution by True Class")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
_p = OUTPUTS_DIR / "calibration_diagram.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\calibration_diagram.png


---
## Section 11 — Error Analysis

Inspects the hardest false positives and false negatives — the samples where
the model was most confidently wrong. These are the cases most worth
reviewing clinically.

- **False positives (FP)**: non-anemic patients predicted as anemic with
  the highest confidence — risk of unnecessary treatment.
- **False negatives (FN)**: anemic patients predicted as non-anemic with
  the highest confidence — risk of missed diagnosis.


In [23]:
# =============================================================================
#  11.1  Identify hardest errors
# =============================================================================
N_TOP = EVAL_CONFIG["top_errors_n"]
threshold = EVAL_CONFIG["threshold"]

fp_mask = (y_test_true == 0) & (y_test_pred == 1)   # non-anemic → predicted anemic
fn_mask = (y_test_true == 1) & (y_test_pred == 0)   # anemic    → predicted non-anemic

# Among FPs, pick the ones with highest predicted probability (most confident)
fp_probs   = y_test_prob[fp_mask]
fp_indices = np.where(fp_mask)[0]
fp_sorted  = fp_indices[np.argsort(-fp_probs)[:N_TOP]]

# Among FNs, pick the ones with lowest predicted probability (most confident)
fn_probs   = y_test_prob[fn_mask]
fn_indices = np.where(fn_mask)[0]
fn_sorted  = fn_indices[np.argsort(fn_probs)[:N_TOP]]

print(f"  Total test samples  : {len(y_test_true)}")
print(f"  False positives (FP): {fp_mask.sum()}  (worst {min(N_TOP, len(fp_sorted))} shown)")
print(f"  False negatives (FN): {fn_mask.sum()}  (worst {min(N_TOP, len(fn_sorted))} shown)")
print()

# Build a summary dataframe from test features
_test_clin = _clin_arr[test_idx]
_test_geo  = _geo_arr[test_idx]

error_summary_rows = []
for idx_type, indices, label in [
    ("FP", fp_sorted, "False Positive"),
    ("FN", fn_sorted, "False Negative"),
]:
    for i in indices:
        row = {
            "error_type"   : label,
            "sample_idx"   : int(i),
            "true_label"   : int(y_test_true[i]),
            "pred_prob"    : float(y_test_prob[i]),
            "pred_label"   : int(y_test_pred[i]),
            "state_risk"   : float(_test_geo[i]),
        }
        # Attach available clinical features
        for col_idx, col_name in enumerate(clinical_features_df.columns):
            row[col_name] = float(_test_clin[i, col_idx])
        error_summary_rows.append(row)

error_df = pd.DataFrame(error_summary_rows)

_p = OUTPUTS_DIR / "error_analysis.csv"
error_df.to_csv(_p, index=False)
print(f"Error analysis CSV saved -> {_p}")
print()
print("Top 5 False Positives (highest confidence non-anemic → predicted anemic):")
print(error_df[error_df["error_type"]=="False Positive"]
      .head(5)[["sample_idx","true_label","pred_prob","state_risk"]].to_string(index=False))
print()
print("Top 5 False Negatives (lowest confidence anemic → predicted non-anemic):")
print(error_df[error_df["error_type"]=="False Negative"]
      .head(5)[["sample_idx","true_label","pred_prob","state_risk"]].to_string(index=False))


  Total test samples  : 15
  False positives (FP): 0  (worst 0 shown)
  False negatives (FN): 4  (worst 4 shown)

Error analysis CSV saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\error_analysis.csv

Top 5 False Positives (highest confidence non-anemic → predicted anemic):
Empty DataFrame
Columns: [sample_idx, true_label, pred_prob, state_risk]
Index: []

Top 5 False Negatives (lowest confidence anemic → predicted non-anemic):
 sample_idx  true_label  pred_prob  state_risk
          8           1   0.327066       0.534
         12           1   0.348098       0.534
          4           1   0.390687       0.534
          2           1   0.466345       0.534


In [24]:
# =============================================================================
#  11.2  Error confidence distribution plot
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Error Analysis — Confidence Distribution of Misclassifications",
             fontsize=12, fontweight="bold")

# ── FP distribution ───────────────────────────────────────────────────────────
ax = axes[0]
if fp_mask.sum() > 0:
    ax.hist(y_test_prob[fp_mask], bins=15, color="#FAA43A", edgecolor="white",
            alpha=0.8, label=f"FP (n={fp_mask.sum()})")
    ax.axvline(y_test_prob[fp_mask].mean(), color="#333", linestyle="--",
               linewidth=1.5, label=f"Mean = {y_test_prob[fp_mask].mean():.3f}")
ax.set_xlabel("Predicted Probability"); ax.set_ylabel("Count")
ax.set_title("False Positives — Confidence Distribution")
ax.legend(); ax.grid(alpha=0.3)

# ── FN distribution ───────────────────────────────────────────────────────────
ax = axes[1]
if fn_mask.sum() > 0:
    ax.hist(y_test_prob[fn_mask], bins=15, color="#5DA5DA", edgecolor="white",
            alpha=0.8, label=f"FN (n={fn_mask.sum()})")
    ax.axvline(y_test_prob[fn_mask].mean(), color="#333", linestyle="--",
               linewidth=1.5, label=f"Mean = {y_test_prob[fn_mask].mean():.3f}")
ax.set_xlabel("Predicted Probability"); ax.set_ylabel("Count")
ax.set_title("False Negatives — Confidence Distribution")
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
_p = OUTPUTS_DIR / "error_confidence_dist.png"
fig.savefig(_p, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved -> {_p}")


Saved -> C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\outputs\error_confidence_dist.png


---
## Section 12 — Final Summary Report

In [25]:
# =============================================================================
#  12.1  Comprehensive final summary
# =============================================================================
divider = "=" * 62

print(divider)
print("       ANEMIAFUSIONNET — PHASE 6: EVALUATION COMPLETE")
print(divider)

# ── Full model test metrics ───────────────────────────────────────────────────
print()
print("  FULL MODEL — Test Set Metrics")
print(f"  {'-'*50}")
for metric, label in [
    ("accuracy",    "Accuracy   "),
    ("precision",   "Precision  "),
    ("recall",      "Recall     "),
    ("specificity", "Specificity"),
    ("f1",          "F1 Score   "),
    ("roc_auc",     "ROC-AUC    "),
    ("mcc",         "MCC        "),
    ("brier",       "Brier Score"),
]:
    val = full_model_metrics.get(metric, float("nan"))
    print(f"    {label} : {val:.4f}")

# ── Ablation deltas — does geo add value? ─────────────────────────────────────
print()
print("  MODALITY CONTRIBUTION (delta F1 from removing each modality)")
print(f"  {'-'*50}")
_f1_full    = full_model_metrics["f1"]
_f1_no_img  = img_only_metrics["f1"]   # no clin/geo → only image survives
_f1_no_clin = clin_only_metrics["f1"]
_f1_no_geo  = no_geo_metrics["f1"]

# Delta = full model F1 - ablation F1
# Positive delta means removing that modality hurt — it was adding value
print(f"    Image modality value  : Δ F1 = "
      f"{_f1_full - clin_only_metrics['f1']:+.4f}  "
      f"(full vs clinical-only)")
print(f"    Clinical modality val : Δ F1 = "
      f"{_f1_full - img_only_metrics['f1']:+.4f}  "
      f"(full vs image-only)")
print(f"    Geo modality value    : Δ F1 = "
      f"{_f1_full - _f1_no_geo:+.4f}  "
      f"(full vs no-geo)")

# ── Output files ─────────────────────────────────────────────────────────────
print()
print("  Output files generated:")
_expected_outputs = [
    "ablation_comparison.csv",
    "ablation_comparison_bar.png",
    "confusion_matrix_all.png",
    "roc_curve_all.png",
    "pr_curve_all.png",
    "classification_report.csv",
    "calibration_diagram.png",
    "error_analysis.csv",
    "error_confidence_dist.png",
]
for fname in _expected_outputs:
    _p = OUTPUTS_DIR / fname
    status = "✓" if _p.exists() else "✗ MISSING"
    print(f"    {status}  {fname}")

print()
print(divider)
print("  ✓ Warm Start Training    (Phase 3 Section 5A)")
print("  ✓ Transformer Fusion     (Phase 4)")
print("  ✓ Three-Stage Training   (Phase 5)")
print("  ✓ Full Evaluation        (Phase 6 Section 3)")
print("  ✓ Ablation Baselines     (Phase 6 Section 4)")
print("  ✓ Confusion Matrices     (Phase 6 Section 6)")
print("  ✓ ROC + PR Curves        (Phase 6 Sections 7-8)")
print("  ✓ Classification Report  (Phase 6 Section 9)")
print("  ✓ Calibration Analysis   (Phase 6 Section 10)")
print("  ✓ Error Analysis         (Phase 6 Section 11)")
print()
print(divider)
print("  Project complete: AnemiaFusionNet — all 6 phases finished.")
print(divider)


       ANEMIAFUSIONNET — PHASE 6: EVALUATION COMPLETE

  FULL MODEL — Test Set Metrics
  --------------------------------------------------
    Accuracy    : 0.7333
    Precision   : 1.0000
    Recall      : 0.5556
    Specificity : 1.0000
    F1 Score    : 0.7143
    ROC-AUC     : 0.9444
    MCC         : 0.5774
    Brier Score : 0.1797

  MODALITY CONTRIBUTION (delta F1 from removing each modality)
  --------------------------------------------------
    Image modality value  : Δ F1 = +0.0186  (full vs clinical-only)
    Clinical modality val : Δ F1 = +0.1810  (full vs image-only)
    Geo modality value    : Δ F1 = +0.0989  (full vs no-geo)

  Output files generated:
    ✓  ablation_comparison.csv
    ✓  ablation_comparison_bar.png
    ✓  confusion_matrix_all.png
    ✓  roc_curve_all.png
    ✓  pr_curve_all.png
    ✓  classification_report.csv
    ✓  calibration_diagram.png
    ✓  error_analysis.csv
    ✓  error_confidence_dist.png

  ✓ Warm Start Training    (Phase 3 Section 5A)
  ✓

In [1]:
from pathlib import Path

IMAGE_DIR = Path(r"C:\Users\sandhya\OneDrive\Desktop\Data Vidwan\AnemiaFusionNet\dataset\images")

count = len(list(IMAGE_DIR.rglob("*.jpg"))) \
      + len(list(IMAGE_DIR.rglob("*.png"))) \
      + len(list(IMAGE_DIR.rglob("*.jpeg")))

print("Total Images:", count)

Total Images: 380
